In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))  

True
NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import re


C:\Users\Aashish\AppData\Local\Temp\ipykernel_23860\1682290108.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
d:\rag chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dir_path = r"data"
VECTOR_DIR = "rag/vectorstore"



In [4]:
def process_all_pdfs(path):
    pdf_files = list(Path(path).rglob("*.pdf"))

    print(f"PDFs found: {len(pdf_files)}")

    all_docs = []

    for index, pdf_file in enumerate(pdf_files, start=1):
        print(f"\n[{index}/{len(pdf_files)}] Processing: {pdf_file.name}")

        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            all_docs.extend(documents)

            print(f"Pages loaded: {len(documents)}")

        except Exception as error:
            print(f"Skipped {pdf_file.name}: {error}")

    print(f"\nTotal pages loaded: {len(all_docs)}")

    return all_docs

In [5]:
import re
from bs4 import BeautifulSoup


def clean_text(text: str) -> str:
    """
    Clean text extracted from agricultural PDFs.
    """

    # Convert non-string values safely
    if not isinstance(text, str):
        return ""

    # Remove HTML/XML tags if present
    text = BeautifulSoup(text, "html.parser").get_text(" ")

    # Remove null characters and unusual control characters
    text = text.replace("\x00", " ")
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", text)

    # Normalize common Unicode characters
    text = text.replace("\u00a0", " ")   # non-breaking space
    text = text.replace("\u2018", "'")
    text = text.replace("\u2019", "'")
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = text.replace("\u2013", "-")
    text = text.replace("\u2014", "-")

    # Join words broken by a PDF line break:
    # "agricul-\nture" -> "agriculture"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # Replace remaining line breaks with spaces
    text = re.sub(r"\s*\n\s*", " ", text)

    # Remove repeated spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove repeated punctuation artifacts
    text = re.sub(r"\.{3,}", "...", text)
    text = re.sub(r"\-{3,}", "---", text)

    # Remove isolated page numbers
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Remove common extraction artifacts
    text = re.sub(r"\bPage\s+\d+\s*(of\s+\d+)?\b", "", text, flags=re.I)

    # Final whitespace cleanup
    text = text.strip()

    return text

In [6]:
def split_documents(docs):
    """
    Splits loaded documents into smaller chunks
    suitable for embedding and retrieval.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            "",
        ],
    )

    chunks = text_splitter.split_documents(docs)
    print(f"Created {len(chunks)} chunks from {len(docs)} documents")

    return chunks



In [7]:
docs = process_all_pdfs(dir_path)

# Clean extracted PDF text
for doc in docs:
    doc.page_content = clean_text(doc.page_content)

# Remove very short or empty documents
docs = [
    doc for doc in docs
    if len(doc.page_content.strip()) > 50
]

chunks = split_documents(docs)

print(f"Cleaned documents: {len(docs)}")
print(f"Created chunks: {len(chunks)}")

print(f"Created {len(chunks)} chunks from {len(docs)} documents.")

PDFs found: 14

[1/14] Processing: 2409.08916v2.pdf
Pages loaded: 310

[2/14] Processing: ICAR Agro advisory for Kharif.pdf
Pages loaded: 2

[3/14] Processing: ICAR En-Kharif Agro-Advisories for Farmers 2025.pdf
Pages loaded: 310

[4/14] Processing: Rabi-Agro-Advisory-2021-22.pdf
Pages loaded: 755

[5/14] Processing: ICAR-technologies-Biofertilizers.pdf
Pages loaded: 56

[6/14] Processing: ICAR-Technologies-Biopesticides.pdf
Pages loaded: 48

[7/14] Processing: Recommended INM Packages -1.pdf
Pages loaded: 36

[8/14] Processing: Updated_140723FinalDraftManualClasses6_8.pdf
Pages loaded: 31

[9/14] Processing: doc2024821378701.pdf
Pages loaded: 4

[10/14] Processing: PIB_2002012_exact.pdf
Pages loaded: 6

[11/14] Processing: Pradhan Mantri Kisan Samman Nidhi.pdf
Pages loaded: 3

[12/14] Processing: ca5789en.pdf
Pages loaded: 81

[13/14] Processing: ca7596en.pdf
Pages loaded: 73

[14/14] Processing: RB-50-Micro-Irrigation-Efficient-Utilization-of-Water-Resources-for-Sustainable-Crop-Prod

In [8]:
chunks[2]

Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'data\\crops\\2409.08916v2.pdf', 'file_path': 'data\\crops\\2409.08916v2.pdf', 'total_pages': 310, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-05-28T04:40:59+00:00', 'trapped': '', 'modDate': 'D:20250528044059Z', 'creationDate': '', 'page': 3}, page_content='Cheena Vyas Concept & Guidance: Dr. Rajbir Singh Compilation: ICAR-Agricultural Technology Application Research Institute, Zone-1, Ludhiana Arvind Kumar, R.R. Burman and Ranjay K. Singh Division of Agricultural Extension, Krishi Anusandhan Bhawan, ICAR, New Delhi DDG (Agricultural Extension), ICAR, New Delhi Layout & Design: Parvender Sheoran and Rajesh K Rana Publisher: Anuradha Agarwal Project Director, Directorate of Knowledge in Agriculture, ICAR, New Delhi May 2025 © 2025 Indian Council of Agricultural Research (ICAR), New Delhi This publication may be used/shared freely for non-commercial

In [9]:
def get_embeddings():
    ef = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True}
    )
    print("Embedding model loaded successfully!")
    return ef

In [10]:
embeddings = get_embeddings()


vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
vectorstore.save_local("faiss_db")
print("FAISS database created successfully!")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24272.88it/s]


Embedding model loaded successfully!
FAISS database created successfully!
